# Spin Orbite model for 3d orbital
## Author: Mathieu Desmarais
### Date: 28-05-2026 

## Packages and modules

In [55]:
import numpy as np 
import qutip as qt                 # Package for quantum mecanic
import ufss as uf                  # Generation of doubled sided feynmann diagram
import matplotlib.pyplot as plt

# 2. La spectroscopie 2D
from qudpy.Classes import System   # Calcul and generation of 2D plot
import qudpy.plot_functions as pf

## Random function definition

In [ ]:
##### Calcul of the dispersion for the spin-Peierls phases #####
def Spin_Peierls_dispersion(k, T, B, J, T_SP_0, delta_0, beta):
    alpha = 0.004
    T_SP = T_SP_0 * (1 - alpha * B**2)
    T_SP = max(0.0,T_SP)

    if T < T_SP: 
        delta = delta_0 * (1 - T/T_SP)**beta
    else: 
        delta=0

    #Cross-Fisher relation: the gap Delta scale like J* delta^(2/3)
    Delta = 2.0 * J * (delta **(2/3)) if delta > 0 else 0.0

    v = (np.pi * J) /2
    epsilon_k = np.sqrt(Delta**2 + (v * np.sin(k)**2))
    return epsilon_k


def get_coupling(k, eta=0.0, a_dimer=1.0, parity='odd'):
    """
    Calculates the effective SOC coupling for a dimerized lattice (e.g., CuGeO3).

    Parameters:
    -----------
    k : float
        Wave vector.
    eta : float
        Mixing parameter. Represents symmetry breaking due to lattice 
        fluctuations, magnetic frustration, or crystal field effects.
    a_dimer : float
        Relative distance between atoms in the dimer (normalized).
    parity : str
        'odd' (antisymmetric -> sin), 'even' (symmetric -> cos), or 'mixed'.

    Returns:
    --------
    float
        The effective coupling strength at wave vector k.
    """
    # The phase shift depends on the internal dimension of the dimer
    phase = k * (a_dimer / 2.0)
    
    if parity == 'odd':
        return np.sin(phase)
    elif parity == 'even':
        return np.cos(phase)
    else:
        # Arbitrary mixing if the crystal symmetry is broken
        return np.sin(phase) + eta * np.cos(phase)
    


def simulate_dynamics_soc(valeurs_lambda_soc, n_modes, tlist):
    """
    Simulates the transfer dynamics for different Spin-Orbit Coupling values.
    
    Returns:
        dict: A dictionary containing the bright and dark populations for each lambda.
    """
    print(f"--- Preparation of base Hamiltonians (N_modes = {n_modes}) ---")
    
    # 1. Calculation outside the loop (Optimization: only calculate once what is independent of lambda)
    H_spin, annihilations_ops, k_values = BuildKSpaceSpinHamiltonian(n_modes, T, B, J, T_SP_0, delta_0, beta, max_bosons_per_mode=2, max_total_triplons=1)
    H_orb, dipole_op, L_op, ket_g, ket_d, ket_b = BuildOrbitalHamiltonian(E_d, E_B)

    dim_orb = H_orb.shape[0]
    dim_spin = H_spin.shape[0]

    # Initial state
    ket_0_spin = qt.basis(dim_spin, 0) 
    psi0 = qt.tensor(ket_b, ket_0_spin)
    psi0 = qt.Qobj(psi0.data, dims=[[dim_orb * dim_spin], [1]])

    # Projectors
    P_b = ket_b * ket_b.dag()
    P_b_tot = qt.tensor(P_b, qt.qeye(dim_spin))
    P_b_tot = qt.Qobj(P_b_tot.data, dims=[[dim_orb * dim_spin], [dim_orb * dim_spin]])

    P_d = ket_d * ket_d.dag()
    P_d_tot = qt.tensor(P_d, qt.qeye(dim_spin))
    P_d_tot = qt.Qobj(P_d_tot.data, dims=[[dim_orb * dim_spin], [dim_orb * dim_spin]])

    # Dictionary to store the results
    resultats = {}

    # 2. Loop over SOC coupling values
    print("--- Start of simulation loop ---")
    for lamda in valeurs_lambda_soc:
        print(f"Simulation in progress for lambda_SOC = {lamda}...")
        
        # Assemble the total Hamiltonian for the current lambda
        H_tot, dipole_tot = TotalHamiltonian(H_orb, dipole_op, L_op, H_spin, annihilations_ops, k_values, lamda)
        
        # Resolution of the von Neumann / Schrödinger equation
        result = qt.mesolve(H_tot, psi0, tlist, e_ops=[P_b_tot, P_d_tot])
        
        pop_bright = result.expect[0]
        pop_dark = result.expect[1]
        
        # Save results
        resultats[lamda] = {
            'bright': pop_bright,
            'dark': pop_dark
        }

        # 3. Graphical display for this lambda
        plt.figure(figsize=(9, 5))
        plt.plot(tlist, pop_bright, label="Population (Bright State)", color="red", linewidth=2)
        plt.plot(tlist, pop_dark, label="Population (Dark State + Triplons)", color="blue", linewidth=2)
        plt.plot(tlist, pop_bright + pop_dark, label="Total Population (Conservation)", color="black", linestyle="--")

        plt.xlabel("Time")
        plt.ylabel("Probability")
        plt.title(f"SOC transfer dynamics ($\lambda_{{SOC}}$ = {lamda}, $N_{{modes}}$ = {n_modes})")
        plt.legend()
        plt.grid(True)
        plt.show()
        
    print("--- Simulations finished ---")
    return resultats

import matplotlib.pyplot as plt

def analyze_system_spectrum(n_modes, params_spin, params_orb, lambda_soc, eta=0.0, a_dimer=1.0, parity='odd'):
    """
    Constructs the Hamiltonians, verifies hermiticity, and plots the energy spectrum.
    
    Args:
        n_modes (int): Number of modes for Brillouin zone discretization.
        params_spin (dict): Parameters for BuildKSpaceSpinHamiltonian.
        params_orb (dict): Parameters for BuildOrbitalHamiltonian.
        lambda_soc (float): Spin-Orbit Coupling strength.
        eta (float): Parity mixing parameter.
        a_dimer (float): Normalized dimer distance.
        parity (str): Symmetry mode ('odd', 'even', 'mixed').
    """
    
    # 1. Hamiltonian Construction
    print("--- Constructing Hamiltonians ---")
    
    # Build the magnetic (spin) sector
    H_spin, annihilations_ops, k_values = BuildKSpaceSpinHamiltonian(
        n_modes, **params_spin
    )
    
    # Build the electronic (orbital) sector
    H_orb, dipole_op, L_op, ket_g, ket_d, ket_b = BuildOrbitalHamiltonian(
        **params_orb
    )
    
    # Build the total coupled Hamiltonian
    # Note: dims_spin is required to ensure consistent Hilbert space structure
    dims_spin = [params_spin.get('max_bosons_per_mode', 2)] * n_modes
    H_tot, dipole_tot = TotalHamiltonian(
        H_orb, dipole_op, L_op, H_spin, annihilations_ops, k_values, 
        lambda_soc, eta=eta, a_dimer=a_dimer, parity=parity
    )
    
    # 2. Hermiticity Verification
    print("\n--- Checking Hermiticity ---")
    print(f"Orbital Hamiltonian is Hermitian: {H_orb.isherm}")
    print(f"Spin Hamiltonian is Hermitian:    {H_spin.isherm}")
    print(f"Total Hamiltonian is Hermitian:   {H_tot.isherm}")
    
    # 3. Energy Spectrum Calculation
    print("\n--- Energy Spectrum ---")
    energies = H_tot.eigenenergies()
    print("Lowest 10 eigenvalues of the system:")
    print(energies[:10])
    
    # 4. Visualization
    plt.figure(figsize=(8, 5))
    plt.plot(energies, 'o', markersize=3, color='teal')
    plt.title("Energy Spectrum of the Coupled System")
    plt.ylabel("Energy (a.u.)")
    plt.xlabel("State Index")
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()


   
# Understanding the Model Structure
This code uses QuTiP (Quantum Toolbox in Python) to simulate a hybrid quantum system. It follows a three-step construction logic:

## 1. Orbital Sector Construction
The model defines a 3-level system ($|g\rangle, |d\rangle, |b\rangle$). The $|b\rangle$ (Bright) state is what the laser interacts with, while the $|d\rangle$ (Dark) state acts as a reservoir for magnetic energy. The orbital Hamiltonian assigns energy levels to these states, and the L_op operator defines how an electron moves between the Bright and Dark orbitals.

## 2. Spin Sector (The "Magic" of enr_destroy)
Simulating a full 1D spin chain in a computer is computationally expensive because the Hilbert space grows exponentially ($2^N$). The code uses qt.enr_destroy, which stands for Energy-Restricted annihilation operators.
- Concept: Instead of allowing every possible configuration, it restricts the total number of "triplons" created in the system (max_total_triplons).
- Benefit: For low-energy physics, this allows you to simulate large systems (many $k$-modes) without crashing your RAM.
## 3. Total Hamiltonian & Coupling
The total system is the tensor product of the Orbital and Spin sectors.
- The Link: The H_SOC_tot loop connects the two. It forces an exchange of energy: when an orbital transitions from Bright to Dark (via L_op), it creates or annihilates a triplon in the spin chain (via S_op_k).

- Momentum dependence: The term np.sin(k) ensures that this coupling respects the parity symmetry of the crystal, which is physically necessary for these types of materials.

## 4. Description input of the Hamiltonian
#### Build OrbitalHamiltonian()
- Delta_dark : Energy of the "dark " orbital state (forbidden by light)
- Delta_bright: Energy of the "bright" orbital state (active optically)

#### BuildKSpaceSpinHamiltonian()
- Parameter -- Type ---- Description
- N_modes -- int ----- Number of points to discretiae the Brillouin zone
- T ----------- float --- System temperature (influence triplon gap)
- B ----------- float --- External magnetic field (Zeeman splitting)
- J ----------- float --- Exchange integral (strength of the spin-spin interaction)
- T_SP_0 ------ float --- Critical temperature of the Spin-Peierls phase
- delta_0 ----- float --- Dimerization parameter (lattice distortion strength)
- beta -------- float --- scaling factor of the dispersion relation
- max_boson_per_mode - int - Max occupancy allowed for a single k-mode
- max_total_triplons - int - Max total triplons in the whole system (Hilbert space cut off)

##### Total Hamiltonian
- H_orb ---- Qobj --- the orbital Hamiltonian
- dipole_op--Qobj --- The dipolar transition operator
- L_op ----- Qobj --- The orbital transition operator for SOC
- H_spin_k - Qobj --- The spin Hamiltonian in k-space
- annihilation_ops - list - The list of annihilation operators from the spin sector
- k_values -------- list -- The momentum discretization array
- lambda_SOC ------- float -- The strength of the Spin orbit coupling


In [142]:
def BuildOrbitalHamiltonian(Delta_dark, Delta_bright):
    """
    Construction of the orbital sector (ground, Bright, Dark).
    """
    ket_g = qt.basis(3, 0) # |ground>
    ket_d = qt.basis(3, 1) # |dark>
    ket_b = qt.basis(3, 2) # |bright>

    # Energy projection operators
    P_d = ket_d * ket_d.dag()
    P_b = ket_b * ket_b.dag()

    # Orbital Hamiltonian (sets energy levels for excited states)
    H_orb = Delta_dark * P_d + Delta_bright * P_b

    # Dipolar Operator (Light-matter coupling: pump excites g -> b)
    dipole_op = ket_g * ket_b.dag() + ket_b * ket_g.dag()

    # Orbital operator for Spin-Orbit Coupling (SOC) (Mixes b <-> d)
    L_op = ket_d * ket_b.dag() + ket_b * ket_d.dag() 

    return H_orb, dipole_op, L_op, ket_g, ket_d, ket_b

def BuildKSpaceSpinHamiltonian(N_modes, T, B, J, T_SP_0, delta_0, beta, max_bosons_per_mode=2, max_total_triplons=1):
    """
    Constructs the triplon sector in k-space.
    Uses 'enr_destroy' to restrict the Hilbert space size for efficiency.
    """
    # Brillouin Zone discretization
    k_values = np.linspace(-np.pi, np.pi, N_modes)
    epsilon_k = [Spin_Peierls_dispersion(k, T, B, J, T_SP_0, delta_0, beta) for k in k_values]
    
    # Dimensions defined by max bosons allowed per mode
    dims = [max_bosons_per_mode] * N_modes
    
    # Energy-Restricted (ENR) annihilation operators
    # If max_total_triplons = 1, the matrix size is linear (N_modes + 1) instead of 2^N
    annihilation_ops = qt.enr_destroy(dims, excitations=max_total_triplons)

    # Construct the free triplon Hamiltonian
    H_spin = 0
    for i in range(N_modes):
        a = annihilation_ops[i]
        H_spin += epsilon_k[i] * a.dag() * a  

    return H_spin, annihilation_ops, k_values

def TotalHamiltonian(H_orb, dipole_op, L_op, H_spin_k, annihilations_ops, k_values, 
                     lambda_SOC, eta=0.0, a_dimer=1.0, parity='odd'):
    
    dim_orb = H_orb.shape[0]
    dim_spin = H_spin_k.shape[0]
    
    # Calculate total dimension for flattening
    dim_tot = dim_orb * dim_spin 
    
    Id_orb = qt.qeye(dim_orb)
    Id_spin = qt.qeye(dim_spin)
    
    H_orb_tot = qt.tensor(H_orb, Id_spin)
    # FLATTEN DIMENSIONS
    H_orb_tot.dims = [[dim_tot], [dim_tot]] 
    
    H_spin_tot = qt.tensor(Id_orb, H_spin_k)
    # FLATTEN DIMENSIONS
    H_spin_tot.dims = [[dim_tot], [dim_tot]] 
    
    # H_SOC_tot construction using the get_coupling function
    H_SOC_tot = 0
    for i, k in enumerate(k_values):
        a_k = annihilations_ops[i]
        
        # Calling the new coupling function
        V_k = get_coupling(k, eta=eta, a_dimer=a_dimer, parity=parity)
        
        S_op_k = V_k * (a_k + a_k.dag())
        
        H_SOC_term = qt.tensor(L_op, S_op_k)
        # FLATTEN DIMENSIONS
        H_SOC_term.dims = [[dim_tot], [dim_tot]] 
        
        H_SOC_tot += lambda_SOC * H_SOC_term

    # Combine and finalize (Now QuTiP will allow the addition)
    H_tot = H_orb_tot + H_spin_tot + H_SOC_tot
    
    dipole_tot = qt.tensor(dipole_op, Id_spin)
    # Flatten dipole as well just to be safe for future operations
    dipole_tot.dims = [[dim_tot], [dim_tot]] 
    
    return H_tot, dipole_tot

## Physics constant and definition of parameter



In [156]:
N_modes=300
T=5
B=0
J=1
delta_0= 0.01
beta= 0.5
Lamda_SOC = 0.20
max_bosons_per_mode = 2
E_d = 0.99
E_B = 1

## Little test on the model
Some preminary test to see if the model is making sens
- First test verify is the hamiltonian are hermitian and plot all the energie state. 
- The second test plot the dynamic of the dark and bright population.


In [ ]:
params_spin_dict = {
    'T': T, 'B': B, 'J': J, 'T_SP_0': T_SP_0, 
    'delta_0': delta_0, 'beta': beta, 
    'max_bosons_per_mode': 2, 'max_total_triplons': 1
}

params_orb_dict = {'Delta_dark': E_d, 'Delta_bright': E_B}
lambda_soc = 0.001
analyze_system_spectrum(300, params_spin_dict, params_orb_dict, lambda_soc=lambda_soc)
    

In [ ]:
Lamda_SOC_test = [0.01,0.02, 0.05]
N_modes_test = 300
tlist_test = np.linspace(0, 1000, 10000)

# Appel de la fonction (elle utilisera les variables globales T, B, J, E_d, etc. définies ailleurs dans ton script)
donnees_populations = simulate_dynamics_soc(Lamda_SOC_test, N_modes_test, tlist_test)

## Dissipation (Lindblad Operator)

## Density Matrix 

## Impulsion sequence

## Feynmann Diagram

## Dynamic calcul

## Fourier Transform

## Signal extraction

## 2D spectra plot